<div align="center">

<h1>RVC — Retrieval-based-Voice-Conversion-WebUI</h1>

<b>Google Colab 一键运行 · One-click notebook for Google Colab</b><br>
简单易用的语音音色转换 / 变声器框架 · Easy-to-use voice conversion framework<br><br>

<a href="https://colab.research.google.com/github/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/RVC_Colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

</div>

## 使用说明 / How to use

1. **选择 GPU 运行时 / Select a GPU runtime**：菜单 `代码执行程序 → 更改运行时类型`（`Runtime → Change runtime type`），硬件加速器选择 **T4 GPU**（或 L4 / A100）。
2. **从上到下依次运行所有单元格 / Run all cells from top to bottom.** 首次完整安装约需 10–15 分钟。
3. 「启动 WebUI」单元格会输出一个 `https://xxxx.gradio.live` 公网链接，点击即可在本地浏览器打开完整的 RVC 训练 / 推理界面。
4. 训练完成后运行「备份到 Google Drive」单元格保存模型——**Colab 磁盘是临时的**。

> ⚠️ **注意 / Notes**
>
> - 本 Notebook 按仓库 README 的安装流程编写：克隆 `main` 分支 → 两阶段安装依赖（CUDA 版 Torch + requirements）→ 从 Hugging Face 下载底模 → `python webui.py --colab` 启动。
> - 训练建议准备 **10 分钟以上**的低底噪干声数据；免费版 T4 足以完成小数据集训练。
> - `gradio.live` 链接只在 Colab 运行时存活期间有效；运行时被回收后需重新运行全部单元格。


In [ ]:
# ================== 可修改参数 / Configurable parameters ==================

# 要克隆的仓库（使用自己的 fork 时请修改）/ Repository to clone
REPO_URL = "https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git"
REPO_BRANCH = "main"

# 项目在 Colab 虚拟机上的安装目录 / Installation directory on the Colab VM
PROJECT_DIR = "/content/Retrieval-based-Voice-Conversion-WebUI"

# v1 底模与 pymss 分离权重默认不下载，如需请在“下载底模”单元格中取消对应注释。
# v1 base models and pymss separation weights are optional: uncomment the
# matching lines in the “download base models” cell if you need them.


In [ ]:
# ================== 检查 GPU 运行时 / Verify the GPU runtime ==================
import subprocess
import sys

print("Python:", sys.version.split()[0])
if sys.version_info[:2] != (3, 12):
    print(
        "⚠️ 本项目面向 Python 3.12，当前为 %d.%d，后续安装可能出现兼容性问题。"
        " This project targets Python 3.12." % sys.version_info[:2]
    )

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise SystemExit(
        "❌ 未检测到 GPU / No GPU detected!\n"
        "请点击菜单「代码执行程序 → 更改运行时类型」选择 T4 GPU，然后从头重新运行所有单元格。\n"
        "Open Runtime → Change runtime type, pick a GPU, then rerun all cells."
    )
print("GPU:", smi.stdout.strip())


In [ ]:
# ================== 克隆仓库并安装系统依赖 / Clone the repo and install system packages ==================
!apt-get -qq update > /dev/null
!apt-get -qq install -y ffmpeg libsndfile1 libportaudio2 unzip > /dev/null
!test -d {PROJECT_DIR}/.git && echo "✅ 仓库已存在，跳过克隆 / repository already cloned" || git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}
!ls {PROJECT_DIR}


In [ ]:
# ================== 安装依赖 / Install dependencies（约 5–10 分钟）==================
# 按仓库 README 两阶段安装，CUDA 版本固定为 cu118：Colab 的 GPU（T4 / L4 / A100）
# 都能运行 cu118 轮子，只有 RTX 50 系（Blackwell，Colab 不提供）才需要 cu128。
# Stage 2 前把 requirements 内置的国内镜像替换为官方源（Colab 海外网络更快）。
# Two-stage install as in the README, pinned to cu118: every Colab GPU
# (T4 / L4 / A100) runs the cu118 wheels; only Blackwell cards need cu128.
!python3 -m pip install torch==2.7.1+cu118 torchaudio==2.7.1+cu118 --index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://pypi.org/simple
!sed -i 's|https://mirrors.pku.edu.cn/pypi/simple|https://pypi.org/simple|g;s|https://mirrors.nju.edu.cn/pytorch/whl/cu118|https://download.pytorch.org/whl/cu118|g' {PROJECT_DIR}/requirments_cu118_py312.txt
!python3 -m pip install -r {PROJECT_DIR}/requirments_cu118_py312.txt
!echo "✅ 依赖安装完成 / dependencies installed"


In [ ]:
# ================== 下载底模 / Download base models（约 5 分钟）==================
# 必需：hubert_base（HuBERT 特征提取）、rmvpe（音高提取）、pretrained_v2（v2 训练底模）、
#       mute.zip（训练用静音样本）。wget -c 支持断点续传，重跑本单元格即可续传。
# 可选：pretrained（v1 底模）、pymss_weights（人声/伴奏分离），见末尾注释行。
!wget -c -q --show-progress -P {PROJECT_DIR}/assets/hubert_base "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base/config.json" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base/preprocessor_config.json" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base/pytorch_model.bin"
!wget -c -q --show-progress -P {PROJECT_DIR}/assets/rmvpe "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt"
!wget -c -q --show-progress -P {PROJECT_DIR}/assets/pretrained_v2 "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/G40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/D40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/G48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/D48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G32k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D32k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/G32k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/D32k.pth"
!wget -c -q --show-progress -P {PROJECT_DIR}/.model-downloads "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/mute.zip"
!python3 -m zipfile -e {PROJECT_DIR}/.model-downloads/mute.zip {PROJECT_DIR}/logs
!echo "✅ 底模下载完成 / base models downloaded"

# 可选下载 / optional downloads：取消注释后重新运行本单元格 / uncomment to use
# !wget -c -q --show-progress -P {PROJECT_DIR}/assets/pretrained "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/f0G40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/f0D40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/G40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/D40k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/f0G48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/f0D48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/G48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/D48k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/f0G32k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/f0D32k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/G32k.pth" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained/D32k.pth"
# !wget -c -q --show-progress -P {PROJECT_DIR}/assets/pymss_weights "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/config_mel_band_roformer_karaoke.yaml" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/dereverb_mel_band_roformer_anvuew.yaml" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/model_bs_roformer_ep_317_sdr_12.9755.ckpt" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/model_bs_roformer_ep_317_sdr_12.9755.yaml" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/model_bs_roformer_ep_368_sdr_12.9628.ckpt" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/model_bs_roformer_ep_368_sdr_12.9628.yaml" "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pymss_weights/model_mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt"


In [ ]:
# ================== 环境自检 / Sanity check ==================
!cd {PROJECT_DIR} && python3 -c "import torch, torchaudio, faiss, gradio; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| gradio:', gradio.__version__)"
!ls {PROJECT_DIR}/assets/hubert_base {PROJECT_DIR}/assets/rmvpe {PROJECT_DIR}/assets/pretrained_v2 | head -20
!ls {PROJECT_DIR}/logs/mute


In [ ]:
# ================== 启动 WebUI / Launch the WebUI ==================
# 等待下方输出 https://xxxx.gradio.live 公网链接后点击使用（--colab 会自动开启
# Gradio 公网分享）。首次启动需加载模型，约 1–3 分钟；链接在运行时存活期间有效，
# 运行本单元格会先停止上一次启动的服务，因此可通过重新运行来重启 / 换新链接。
# Wait for the https://xxxx.gradio.live link printed below, then open it in your
# local browser (--colab turns on Gradio public sharing). Rerunning this cell
# stops the previous server first, so use it to restart or refresh the link.
import os
import re
import subprocess
import time

os.chdir(PROJECT_DIR)
LOG_PATH = os.path.join(PROJECT_DIR, "webui.log")

if "WEBUI_PROC" in globals() and WEBUI_PROC and WEBUI_PROC.poll() is None:
    WEBUI_PROC.terminate()
    time.sleep(3)

webui_log = open(LOG_PATH, "w")
WEBUI_PROC = subprocess.Popen(
    ["python3", "webui.py", "--colab"],
    stdout=webui_log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR,
)

print("🚀 正在启动 WebUI（加载模型约需 1–3 分钟）...")
print("🚀 Starting the WebUI; the public link appears once Gradio is ready ...")
share_url = None
deadline = time.time() + 600
while time.time() < deadline:
    if WEBUI_PROC.poll() is not None:
        break
    time.sleep(5)
    try:
        with open(LOG_PATH, encoding="utf-8", errors="ignore") as f:
            text = f.read()
    except FileNotFoundError:
        continue
    match = re.search(r"https://[\w-]+\.gradio\.live", text)
    if match:
        share_url = match.group(0)
        break

if share_url:
    print()
    print("=" * 62)
    print("🎉 WebUI 已启动 / WebUI is running:", share_url)
    print("=" * 62)
    print("ℹ️ 链接在 Colab 运行时存活期间有效；失效或想重启服务时重新运行本单元格。")
    print("ℹ️ Valid while the runtime is alive; rerun this cell to restart.")
else:
    print("❌ 未能获取公网链接，最近日志 / failed to get the share URL, recent log:")
    with open(LOG_PATH, encoding="utf-8", errors="ignore") as f:
        print(f.read()[-3000:])


## 🖥️ WebUI 使用指南 / WebUI guide

打开上面的 `gradio.live` 链接后（界面语言跟随浏览器 locale，可在顶部下拉切换）：

**模型推理 / Inference**
1. 「模型推理」选项卡 → 点击刷新选择 `assets/weights/` 中的模型（训练完成会自动出现）。
2. 上传 / 输入待转换音频 → 调整变调、检索特征比例等参数 → 点击转换，输出在 `output/` 目录。

**训练 / Training（单说话人）**
1. 先运行下面的「准备训练集」单元格，把干声上传到 `dataset` 目录（或使用 Google Drive 目录）。
2. 「训练」选项卡填写实验名，训练集目录填 `dataset`。
3. 依次点击：①配置实验与数据切分 → ②提取 F0 与 HuBERT 特征 → ③训练模型 → ④训练索引；或直接用「一键训练」。
4. 完成后推理模型在 `assets/weights/`，检索索引在 `assets/indices/`。

**多说话人 / Multi-speaker**：训练模式选择「多说话人」，训练集根目录使用 `名称_ID_重复次数` 子目录结构，详见 [`docs/cn/cli.md`](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/docs/cn/cli.md)。

> 💡 Colab 磁盘是临时的：训练完成后务必运行后面的「备份到 Google Drive」单元格。


In [ ]:
# ================== 准备训练集 / Prepare the training dataset ==================
# 方式一：直接把音频文件上传到下面的目录（左侧文件面板拖拽上传，或代码里解压压缩包）。
# 方式二：把 MOUNT_GDRIVE 改为 True 挂载 Google Drive，训练集目录直接填 Drive 路径。
# Option 1: upload audio files into the folder below via the file browser.
# Option 2: set MOUNT_GDRIVE = True and point the WebUI to a Google Drive folder.
import os

MOUNT_GDRIVE = False
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")

os.makedirs(DATASET_DIR, exist_ok=True)
if MOUNT_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Drive 已挂载，训练集目录可填 /content/drive/MyDrive/ 下的任意文件夹")
    print("✅ Drive mounted; use any folder under /content/drive/MyDrive/")
else:
    print("📁 请把训练音频上传到 / Upload your training audio to:", DATASET_DIR)
    print("   WebUI「输入训练文件夹路径」填 dataset 即可")


In [ ]:
# ================== 备份到 Google Drive / Backup to Google Drive ==================
# 把 BACKUP_TO_DRIVE 改为 True 后运行：复制推理模型（assets/weights）、
# 检索索引（assets/indices）到 Drive；如需断点续训，请再手动备份 logs/<实验名>/。
# Set BACKUP_TO_DRIVE = True to copy trained models and indices to your Drive.
# To resume training later, also back up logs/<experiment>/ manually.
import os
import shutil
import time

BACKUP_TO_DRIVE = False

if BACKUP_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    target = "/content/drive/MyDrive/RVC_backup_%s" % time.strftime("%Y%m%d_%H%M%S")
    os.makedirs(target, exist_ok=True)
    for rel in ["assets/weights", "assets/indices"]:
        src_path = os.path.join(PROJECT_DIR, rel)
        if os.path.isdir(src_path) and os.listdir(src_path):
            dst = os.path.join(target, os.path.basename(src_path))
            shutil.copytree(src_path, dst, dirs_exist_ok=True)
            print("📦 已备份 / backed up:", rel, "->", dst)
    print("✅ 备份完成 / backup finished:", target)
else:
    print("（未启用备份；将 BACKUP_TO_DRIVE 改为 True 后重新运行本单元格 / disabled）")


In [ ]:
# ================== （可选）命令行离线推理 / Optional: offline CLI inference ==================
# 不经过 WebUI，用 infer/cli.py 直接批量转换；全部参数见 docs/cn/cli.md。
# Batch-convert without the WebUI; see docs/en/cli.md for every option.
import glob
import os
import subprocess

print("当前可用模型 / available models:")
for path in sorted(glob.glob(os.path.join(PROJECT_DIR, "assets", "weights", "*.pth"))):
    print("  -", os.path.basename(path))

RUN_CLI_INFERENCE = False
if RUN_CLI_INFERENCE:
    MODEL_NAME = "your-model.pth"  # assets/weights/ 下的文件名 / filename in assets/weights
    INPUT_PATH = "/content/input"  # 音频文件或目录 / audio file or directory
    subprocess.run([
        "python3", "infer/cli.py",
        "--model", "assets/weights/" + MODEL_NAME,
        "--input", INPUT_PATH,
        "--output", "/content/output",
        "--pitch", "0", "--f0-method", "rmvpe", "--index-rate", "0.75",
        "--format", "wav", "--recursive", "--overwrite",
    ], cwd=PROJECT_DIR)


## ❓ 常见问题 / FAQ

- **公网链接打不开或失效？** 重新运行「启动 WebUI」单元格获取新链接；Colab 运行时被回收后需从头重跑全部单元格。
- **训练要多久？** T4 上约 10 分钟的数据集 × 200 epoch 大约 1–2 小时；显存不足（OOM）时调小 batch size。
- **转换出现哑音 / 电音？** 提高「检索特征比例」、使用 `rmvpe` 音高算法、更换 f0 转换档位，或补充训练数据。
- **如何长期保存模型？** 使用「备份到 Google Drive」单元格；`assets/weights/*.pth` 与 `assets/indices/*.index` 是推理所需的最小集合。
- **命令行训练 / 离线推理的全部参数？** 见 [`docs/cn/cli.md`](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/docs/cn/cli.md)（训练五阶段、多说话人清单与 `infer/cli.py`）。
- **更多帮助？** 仓库 [README](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI) 与 [常见问题解答 Wiki](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/wiki/%E5%B8%B8%E8%A7%81%E9%97%AE%E9%A2%98%E8%A7%A3%E7%AD%94)。
